## OOP-Prinzipien mit Code-Beispielen

### 3 Grundprinzipien

Nach Lahres et al. (2021, S. 27–37)  
Lahres, B., Raýman, G., & Strich, S. (2021). Objektorientierte Programmierung: Das umfassende Handbuch (5., aktualisierte Auflage). Rheinwerk Verlag.

#### 🔐 1. Datenkapselung

**Prinzip:**
Interne Daten eines Objekts werden nicht direkt manipuliert, sondern ausschließlich über klar definierte Methoden (z.B. *abschliessen()*) verändert oder über Properties (*ist_aktiv()*) abgefragt.

In [ ]:
@dataclass                                                               # type: ignore
class Einschreibung:
    status: StatusEinschreibung                                          # type: ignore

    def abschliessen(self, enddatum: date, notenschnitt: float) -> None: # type: ignore
        self.end_datum = enddatum
        self.abschluss_note = notenschnitt
        self.status = StatusEinschreibung.ABGESCHLOSSEN                  # type: ignore

UI entscheidet, WANN sich der Status ändert:

In [ ]:
if st.button("Studium abschließen"):                            # type: ignore
    workflow.studium_abschliessen(student_id, studiengang_id)   # type: ignore

# keine direkte Änderung, wie dies:
einschreibung.status = StatusEinschreibung.ABGESCHLOSSEN        # type: ignore

Service koordiniert den Ablauf. 
Es lädt das Model,
ruft fachliche Methoden auf,
speichert das Ergebnis: 

In [ ]:
def studium_abschliessen(
    self,
    *,
    student_id: int,
    studiengang_id: int,
) -> None:
    einschreibung = self._einschreibung_repo.get_aktive(
        student_id, studiengang_id
    )

    notenschnitt = self._pruefung_repo.notenschnitt(student_id, studiengang_id)

    einschreibung.abschliessen(
        enddatum=date.today(), # type: ignore
        notenschnitt=notenschnitt,
    )

    self._einschreibung_repo.update(einschreibung)

Das Model setzt den Status über den Aufruf der Model-Methode:

In [ ]:
einschreibung.abschliessen(...) # type: ignore

<br></br>

---

#### 🔁 2. Polymorphie

**Prinzip:**
Unterschiedliche Implementierungen können über dieselbe Schnittstelle genutzt werden.
Der aufrufende Code kennt nur das Interface, nicht die konkrete Klasse.  

Unterschied Polymorphie - Vererbung:  
Polymorphie beschreibt wie Objekte über eine gemeinsame Schnittstelle verwendet werden  
Vererbung beschreibt Beziehung und gemeinsame Struktur zwischen Klassen

In [ ]:
# Basisklasse, die nur festlegt, WAS möglich ist:
# Services programmieren gegen den Vertrag, nicht gegen die Technik
class KursRepository(ABC):                              # type: ignore
    @abstractmethod                                     # type: ignore
    def create(self, kurs: Kurs) -> int:                # type: ignore
        ...

# Konkrete Implementierung mit SQLite:
# erfüllt den Vertrag create()
# entscheidet WIE der Kurs erstellt/gespecihert wird
class SQLiteKursRepository(KursRepository):
    def create(self, kurs: Kurs) -> int:                # type: ignore
        ...
        
# polymorphe Nutzung im Service:
# Service weiß nicht, ob SQLite genutzt wird
# Er weiß nur: „Ich habe ein Objekt, das sich wie ein KursRepository verhält.“
class WorkflowService:
    def __init__(self, kurs_repo: KursRepository):
        self._kurse = kurs_repo

        def kurs_hinzufuegen(...) -> int:               # type: ignore
            ...
            kurs_id = self._kurse.create(kurs)          # type: ignore

Die Implementierung mit SQLite bleibt damit leicht austauschbar. Kein Code im WorkflwoService muss dafür angepasst werden

In [ ]:
# Heute: SQLite:
repo = SQLiteKursRepository(provider)                   # type: ignore

# Morgen Postgres:
repo = PostgresKursRepository(provider)                 # type: ignore

# Oder für Tests:
repo = InMemoryKursRepository()                         # type: ignore
workflow = WorkflowService(repo)

<br></br>

---

#### 🧬 3. Vererbung

**Prinzip:** Gemeinsames Verhalten / gemeinsame Struktur wird in einer Basisklasse definiert, und konkrete Unterklassen erben davon und ergänzen/überschreiben Details.  

Das zuvor genannte Beispiel zeigt ebenso eine Vererbung.  
- SQLiteKursRepository erbt von KursRepository
- KursRepository gibt gemeinsame Struktur vor
- die abstrakte Methode create() muss in Unterklasse (wie SQLiteKursRepository) implementiert werden

Unterschied Polymorphie - Vererbung:  
Polymorphie beschreibt wie Objekte über eine gemeinsame Schnittstelle verwendet werden  
Vererbung beschreibt die Beziehung und gemeinsame Struktur zwischen Klassen

<br></br>

---
---

### 7 Prinzipien 

Nach Lahres et al. (2021, S. 39–64)  
Lahres, B., Raýman, G., & Strich, S. (2021). Objektorientierte Programmierung: Das umfassende Handbuch (5., aktualisierte Auflage). Rheinwerk Verlag.

#### 1️⃣ Einzige Verantwortung (Single Responsibility, SRP)
Jede Klasse hat genau eine klar definierte Aufgabe, Beispiele im Code:
| Klasse                | Verantwortung                | 
|-----------------------|------------------------------| 
| Kurs	                | Domänenzustand Kurs          | 
| KursRepository	    | Persistenz von Kursen        | 
| WorkflowService	    | Fachlogik / Workflows        | 
| ProgressService	    | Auswertungen                 | 
| ViewModelBuilder	    | UI-nahe Aufbereitung         | 
| UI-Komponenten	    | Darstellung                  |

<br></br>

---

#### 2️⃣ Trennung von Anliegen (Separation of Concerns)
Aufgaben sind klar auf Schichten verteilt, sodass jede Schicht unabhängig geändert oder erweitert werden kann.

| Klasse                            | Verantwortung                | 
|-----------------------------------|------------------------------| 
| UI-Komponenten, Pages	            | Präsentationsschicht         | 
| ViewModelBuilder, ViewModels	    | Präsentations-Vorbereitung   | 
| WorkflowService 	                | Anwendungsschicht            | 
| ProgressService, DTOs	            | Analyse-/Auswertungsschicht  | 
| Repositories	                    | Datenzugriffschicht          | 
| Models	                        | Domänenschicht               | 

<br></br>

---


#### 3️⃣ Wiederholungen vermeiden (Don't repeat yourself, DRY)
Funtionalitäten nur einmal definieren

Beispiele im Code:  
- abgeschlossene_bearbeitungen() für den Abruf von Bearbeitungen mit Status 'abgeschlossen' zentral im WorkflowService
- _context() zum laden der Daten und Speichern im Cache, um wiederholte Datenbankabfrage zu reduzieren

<br></br>

---

#### 4️⃣ Offen für Erweiterung, geschlossen für Änderung (Open-Closed-Principle, OCP)
Neue Funktionen lassen sich ohne Änderungen des bestehenden Codes erstellen

Beispiele:  

In [ ]:
# Neues UI-Backend
get_dashbaord_renderer("cli")           # type: ignore

# Neue DB
PostgresKursRepository(KursRepository)  # type: ignore

- Neue Kacheln:  
    - neues DTO
    - neues ViewModel
    - neue UI-Komponenten

<br></br>

---

#### 5️⃣ Trennung von Schnittstelle & Implementierung (Program to Interfaces)
Code hängt von Interfaces, nicht von konkreten Klassen ab

Beispiele:
- KursRepository & SQLiteKursRepository
- ConnectionProvider & SQLiteConnectionProvider
- UIAdapter & Streamlit-Renderer

<br></br>

---

#### 6️⃣ Umkehr der Abhängigkeiten (Dependency Inversion)
High-Level-Code kennt keine Low-Level-Details

Beispiel:

In [ ]:
class ProgressService:
    def __init__(self, workflow: WorkflowService):
        self._workflow = workflow

Der ProgressService hat:
- keinen Zugriff auf Repsoitories
- keine DB-Kenntnis
- Abhängigkeit nur zur Abstraktion

<br></br>

---

#### 7️⃣ Mach es testbar (Testability)

Logik muss isoliert testbar sein

Beispiele:
- Repositories sind austauschbar
- Services sind ohne UI/DB-Abhängigkeiten
- DTOs & ViewModels sind reine Daten
- keine Logik im UI

<br></br>

---
---

### SOLID Prinzipien

Robert C. Martin. (2000). Design Principles and Design Patterns. Verfügbar 17. Oktober 2025
unter https://objectmentor.com/resources/articles/Principles_and_Patterns.pdf

#### **S** - Einzige Verantwortung (Single Responsibility, SRP)

Jede Klasse hat genau eine klar definierte Aufgabe, Beispiele im Code:
| Klasse                | Verantwortung                | 
|-----------------------|------------------------------| 
| Kurs	                | Domänenzustand Kurs          | 
| KursRepository	    | Persistenz von Kursen        | 
| WorkflowService	    | Onboarding, CRUD-Operationen, Actionbar-Workflows, Queries        | 
| ProgressService	    | Auswertungen                 | 
| ViewModelBuilder	    | UI-nahe Aufbereitung         | 
| UI-Komponenten	    | Darstellung                  |

<br></br>
**Hinweis:**  
WorklfowService ist derziet nicht ausreichend auf 1 Verantwortung ausgerichtet. Es sollte besser aufgeteilt werden, z.B. folgendermaßen:

In [ ]:
# core/workflow/onboarding_service.py
class OnboardingService:
    """Verantwortlich für Ersteinrichtung von Studenten."""
    
    def onboarding(self, *, student_name: str, matrikelnummer: str, ...) -> tuple[int, int, int]: # type: ignore
        ...


# core/workflow/student_query_service.py
class StudentQueryService:
    """Read-Only Queries für Studenten-Daten."""
    
    def student_by_id(...) -> Optional[Student]:                    # type: ignore
        ...
    
    def studiengaenge_by_student_id(...) -> List[Studiengang]:      # type: ignore
        ...
    
    def kurse_fuer_student(...) -> List[Kurs]:                      # type: ignore
        ...


# core/workflow/bearbeitung_workflow_service.py
class BearbeitungWorkflowService:
    """Workflow-Operationen für Bearbeitungen."""
    
    def kurs_hinzufuegen(...) -> int:                               # type: ignore
        ...
    
    def bearbeitung_starten(...) -> None:                           # type: ignore
        ...
    
    def pruefung_abgeben(...) -> None:                              # type: ignore
        ...
    
    def note_eintragen(...) -> Pruefung:                            # type: ignore
        ...

#### **O** - Offen für Erweiterung, geschlossen für Änderung (Open-Closed-Principle, OCP)

Neue Funktionen lassen sich ohne Änderungen des bestehenden Codes erstellen

Beispiele:  

In [ ]:
# Neues UI-Backend
get_dashbaord_renderer("cli")           # type: ignore

# Neue DB
PostgresKursRepository(KursRepository)  # type: ignore

- Neue Kacheln:  
    - neues DTO
    - neues ViewModel
    - neue UI-Komponenten
<br></br>

---

#### **L** - Liskovsche Substitutionsprinzip (Liskov Substitution Principle, LSP)

Unterklassen bzw. alternative Implementierungen müssen sich wie die Basisklasse verhalten und dürfen den Vertrag nicht brechen.
Objekte einer Oberklasse (bzw. Implementierungen eines Interfaces) müssen durch andere Implementierungen ersetzbar sein, ohne dass sich das korrekte Verhalten des Systems ändert.

Beispiel (ProgressService im Test):
Der ViewModelBuilder erstellt aus den Daten des ProgressService ein StudienzieleViewModel. Dafür erwartet er, dass studienziele_daten() entweder

- None (keine Einschreibung) oder
- ein DTO-Objekt mit den Attributen ziel_notenschnitt und ziel_enddatum
zurückgibt.

**Vertragsbruch (LSP-Verstoß):** Ein Fake-ProgressService gibt statt des DTO ein dict zurück. Dadurch schlagen Attributzugriffe wie daten.ziel_enddatum fehl und der Code verhält sich anders als mit der echten Implementierung.

In [ ]:
# viewmodel_builder.py (vereinfacht)
daten = self.progress.studienziele_daten(student_id, studiengang_id)                # type: ignore
ziel_enddatum = daten.ziel_enddatum  # erwartet Attributzugriff


# DTO wie vom ProgressService erwartet 
@dataclass                                                                          # type: ignore
class StudienzieleDaten:
    ziel_notenschnitt: Optional[float]                                              # type: ignore
    ziel_enddatum: Optional[date]                                                   # type: ignore


# ❌ Falscher Fake, der den Vertrag bricht: dict statt DTO-Objekt
class FakeProgressServiceWrong:
    def studienziele_daten(self, student_id: int, studiengang_id=None):
        return {"ziel_notenschnitt": 2.0, "ziel_enddatum": None}



# ✅ korrekter Fake, der den Vertrag einhält: liefert DTO-Objekt (oder None)
class FakeProgressServiceRight:
    def studienziele_daten(self, student_id: int, studiengang_id=None):
        return StudienzieleDaten(ziel_notenschnitt=2.0, ziel_enddatum=None)

<br></br>

---

#### **I** - Schnittstellenaufteilungsprinzip (Interface Segregation Principle, ISP)

Mehrere kleine Interfaces statt ein riesiges. Klassen sollen nicht unnötig Methoden implementieren, die sie nicht benötigen.

**Beispiel (was aktuell noch nicht gut entsprechend dem Prinzip läuft):**  
- ProgressService bietet Vorbereitungen für 7 verschiedene Kacheln. Clients, die nur den Notenverlauf brauchen, müssen die gesamte ProgressSerivce-Klasse mit allen Dependencies kennen.
- ViewModelBuilder ist für alle ViewModels zuständig. Dadurch muss jede UI-Komponente den gesamten Builder injecten, auch wenn sie nur 1 Methode braucht.

ProgressService besser aufteilen auf die jeweiligen fachlichen Bereiche/Kacheln:

In [ ]:
# core/progress/studienziele_service.py
class StudienzieleService:
    """Verantwortlich für Studienziele und deren Status"""
    def studienziele_daten(...) -> Optional[StudienzieleDaten]:                 # type: ignore
        ...
    def studienziele_status_daten(...) -> Optional[StudienzieleStatusDaten]:    # type: ignore
        ...


# core/progress/status_uebersicht_service.py
class StatusUebersichtService:
    """Verantwortlich für die Gesamtübersicht (ECTS, Noten, Tempo)"""
    def status_uebersicht_daten(...) -> Optional[StatusUebersichtDaten]         # type: ignore


# core/progress/verlauf_service.py
class VerlaufService:
    """Verantwortlich für zeitliche Verläufe und Charts."""
    def burndown_daten(...) -> BurndownDaten:                                   # type: ignore
        ...
    
    def notenverlauf_daten(...) -> NotenverlaufDaten:                           # type: ignore
        ...
    
    def bearbeitungsverlauf_daten(...) -> BearbeitungsverlaufDaten:             # type: ignore
        ...


# core/progress/kursplan_service.py
class KursplanService:
    """Verantwortlich für Kursplanung und Gantt-Darstellung."""
    def kursplan_daten(...) -> KursplanDaten:                                   # type: ignore
        ...

ViewModelBuilder besser ebenfalls entsprechend aufteilen:

In [ ]:
# ui/viewmodels/studienziele_builder.py
class StudienzieleViewModelBuilder:
    def __init__(self, studienziele_service: StudienzieleService):
        self._service = studienziele_service
    
    def build_studienziele(...) -> StudienzieleViewModel:                       # type: ignore
        ...
    
    def build_studienziele_status(...) -> StudienzieleStatusViewModel:          # type: ignore
        ...

# ui/viewmodels/status_uebersicht_builder.py
class StatusUebersichtViewModelBuilder:
    def __init__(self, status_uebersicht_service: StatusUebersichtService):     # type: ignore
        self._service = status_uebersicht_service
    
    def build_status_uebersicht() -> StatusUebersichtViewModelBuilder:          # type: ignore


# ui/viewmodels/verlauf_builder.py
class VerlaufViewModelBuilder:                                                  # type: ignore
    def __init__(self, verlauf_service: VerlaufService):                        # type: ignore
        self._service = verlauf_service
    
    def build_notenverlauf(...) -> NotenverlaufViewModel:                       # type: ignore
        ...
    
    def build_bearbeitungsverlauf(...) -> BearbeitungsverlaufViewModel:         # type: ignore
        ...
    
    def build_burndown_chart(...) -> BurndownViewModel:                         # type: ignore
        ...

# ui/viewmodels/kursplan_builder.py
class KursplanViewModelBuilder:
    def __init__(self, kursplan_service: KursplanService):                      # type: ignore
        self._service = kursplan_service
    
    def build_kursplan(...) -> KursplanViewModelBuilder:                        # type: ignore

<br></br>

---

#### **D** Prinzip der Abhängigkeitsumkehr (Dependency Inversion Principle, DIP)

High-Level-Code kennt keine Low-Level-Details

Beispiel:

In [ ]:
class ProgressService:
    def __init__(self, workflow: WorkflowService):
        self._workflow = workflow

ProgressService hat:
- keinen Zugriff auf Repsoitories
- keine DB-Kenntnis
- Abhängigkeit nur zur Abstraktion

<br></br>

---
---